# Prepare the notebook runtime

The implementation notebooks contain the source for the training programs, shared modules and tests. This cell makes local importable copies from their tagged code cells. The walkthroughs run it automatically. Run it once before using the full GPU training commands.

In [ ]:
from pathlib import Path
import os
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks/code").is_dir() and (p / "results/fits.json").is_file())
os.chdir(root)
import json
from tempfile import NamedTemporaryFile

def prepare_runtime(root):
    marker = "# Generated from an implementation notebook. Edit the notebook to change this file.\n"
    for notebook in sorted((root / "notebooks/code").rglob("*.ipynb")):
        document = json.loads(notebook.read_text())
        modules = {}
        for cell in document["cells"]:
            if "module-source" in cell["metadata"].get("tags", []):
                path = cell["metadata"].get("runtime_path") or document["metadata"]["runtime_path"]
                modules.setdefault(path, []).append("".join(cell["source"]))
        if not modules:
            raise ValueError(f"Missing source cells in {notebook.name}")
        for path, parts in modules.items():
            relative = Path(path)
            if relative.is_absolute() or ".." in relative.parts or relative.parts[0] not in {"src", "scripts", "tests"}:
                raise ValueError(f"Unexpected runtime path in {notebook.name}")
            target = root / relative
            content = marker + "\n\n".join(parts) + "\n"
            compile(content, str(target), "exec")
            if target.exists():
                old = target.read_text()
                if old == content:
                    continue
                if not old.startswith(marker):
                    # Existing checkouts can still have a previous standalone source file.
                    import ast
                    if ast.dump(ast.parse(old)) != ast.dump(ast.parse(content)):
                        raise FileExistsError(f"Back up your local changes before regenerating {target}")
            target.parent.mkdir(parents=True, exist_ok=True)
            with NamedTemporaryFile("w", dir=target.parent, delete=False) as tmp:
                tmp.write(content)
            Path(tmp.name).replace(target)
    source = str(root / "src")
    if source not in sys.path:
        sys.path.insert(0, source)

prepare_runtime(root)